In [1]:
!pip install --break-system-packages clickhouse-connect

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 640.6/640.6 kB 5.4 MB/s eta 0:00:00


In [10]:
import clickhouse_connect
import pandas as pd
import os
pd.options.display.max_colwidth = 100
database = 'default'

client = clickhouse_connect.get_client(
    host='localhost', 
    port=8123, 
    username='default', 
    password=''
    )


In [4]:
client.command(f'''
    create table {database}.test1
    (
        order_id UInt8,
        user_id UInt8,
        order_date DateTime,
        total_amount UInt8,
        paid Bool,
        items Array(UInt32)

    )
    engine = MergeTree
    order by order_id

''')

In [8]:
client.command(f''' 
    insert into {database}.test1 values
    (1, 100, '2024-05-10 14:30:00', 1000, 1, [11, 21, 6])

''')

In [9]:
client.query_df(f''' 
select * from {database}.test1
''')

,order_id,user_id,order_date,total_amount,paid,items
0,1,100,2024-05-10 14:30:00+03:00,232,True,"[11, 21, 6]"
1,1,100,2024-05-10 14:30:00+03:00,232,True,"[11, 21, 6]"
2,1,100,2024-05-10 14:30:00+03:00,232,True,"[11, 21, 6]"


In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.mt;
''')

client.command(f'''
    CREATE TABLE IF NOT EXISTS {database}.mt
    (
        id UInt32,
        dt datetime
    )
    ENGINE = MergeTree                                                 -- обязательно нужно указывать движок
    ORDER BY(id)                                                   -- обязательно должны быть колонки в порядке из primary key
    TTL dt + INTERVAL 1 MONTH DELETE;                                  -- от времени dt отсчитывается 1 месяц после чего данные удаляются (данные удаляются полсе слияния)
        --dt + INTERVAL 1 MONTH DELETE WHERE toDayOfWeek(d) IN (6, 7); -- удаление с фильтрацией
        --dt + INTERVAL 1 WEEK TO VOLUME 'aaa',                        -- перемещение данных в вольюм(совокупность дисков, задаваемая в конфигах)
        --dt + INTERVAL 2 WEEK TO DISK 'bbb';                          -- перемещение данных на диск (указывается имя диска)
''')


In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.test_fields_without_ttl;
''')


client.command(f'''
    CREATE TABLE {database}.test_fields_without_ttl
    (
          col_default UInt64 DEFAULT 42                         -- значение по умолчанию
        , col_materialized UInt64 MATERIALIZED col_default * 33 -- к данной колонке можно обратиться только по имени
        , col_alias UInt64 ALIAS col_default + 1                -- к данной колонке можно обратиться только по имени
        , col_codec String CODEC(ZSTD(10))                      -- кодек сжатия
        , col_comment Date COMMENT 'Some comment'               -- комментарий к колонке
    )
    ENGINE = Log;
''')